# 01 — Data Exploration

**NREL PVDAQ Site 4 — Solar PV Power Prediction**

This notebook explores the real-world solar PV dataset from NREL before any
modelling is done.

## Before running this notebook:
```bash
python scripts/download_data.py
python src/preprocessing.py
```

## Contents
1. Load the dataset
2. Inspect shape, columns, dtypes
3. Missing value analysis
4. Date range and sampling frequency
5. Distribution plots (irradiance, power, temperature)
6. Time-series plots
7. Day/night analysis
8. Correlation matrix
9. Irradiance vs Power scatter
10. Seasonal patterns
11. Observations and conclusions

In [ ]:
import sys
from pathlib import Path

# Make src importable
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.config import CLEAN_CSV, TARGET_COL, FEATURE_COLS_ORDERED, NIGHTTIME_IRRADIANCE_THRESHOLD
from src.data_loader import load_clean, dataset_summary

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (14, 4)
print('Libraries loaded.')

In [ ]:
# Load the cleaned dataset
df = load_clean()
print(dataset_summary(df))

In [ ]:
# --- 2. Shape and dtypes ---
print(f'Shape: {df.shape}')
df.dtypes

In [ ]:
# --- 3. Missing values ---
missing = df.isnull().sum()
missing_pct = 100 * missing / len(df)
pd.DataFrame({'count': missing, 'pct': missing_pct}).query('count > 0')

In [ ]:
# --- 4. Date range ---
print(f"From: {df['timestamp'].min()}")
print(f"To:   {df['timestamp'].max()}")
print(f"Span: {df['timestamp'].max() - df['timestamp'].min()}")
gaps = df['timestamp'].diff().dropna()
print(f"Typical interval: {gaps.mode()[0]}")
print(f"Max gap: {gaps.max()}")

In [ ]:
# --- 5. Distribution plots ---
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ['irradiance', 'power', 'ambient_temperature', 'panel_temperature']):
    if col in df.columns:
        ax.hist(df[col].dropna(), bins=60, edgecolor='white', linewidth=0.3)
        ax.set_title(col)
        ax.set_xlabel(col)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6. Power over time ---
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df['timestamp'], df['power'], linewidth=0.3, alpha=0.7)
ax.set_xlabel('Time')
ax.set_ylabel('AC Power (kW)')
ax.set_title('Actual PV Power Over Time')
plt.tight_layout()
plt.show()

In [ ]:
# --- 7. Irradiance over time ---
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df['timestamp'], df['irradiance'], linewidth=0.3, alpha=0.7, color='orange')
ax.set_xlabel('Time')
ax.set_ylabel('Irradiance (W/m²)')
ax.set_title('Solar Irradiance Over Time')
plt.tight_layout()
plt.show()

In [ ]:
# --- 8. Day/night split ---
daytime = df[df['irradiance'] > NIGHTTIME_IRRADIANCE_THRESHOLD]
nighttime = df[df['irradiance'] <= NIGHTTIME_IRRADIANCE_THRESHOLD]
print(f'Daytime rows:  {len(daytime):,}  ({100*len(daytime)/len(df):.1f}%)')
print(f'Nighttime rows: {len(nighttime):,}  ({100*len(nighttime)/len(df):.1f}%)')
print(f'Daytime mean power:  {daytime["power"].mean():.4f} kW')
print(f'Nighttime mean power: {nighttime["power"].mean():.4f} kW')

In [ ]:
# --- 9. Irradiance vs Power scatter (daytime only) ---
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(daytime['irradiance'], daytime['power'], s=1, alpha=0.3)
ax.set_xlabel('Irradiance (W/m²)')
ax.set_ylabel('AC Power (kW)')
ax.set_title('Irradiance vs PV Power (daytime only)')
plt.tight_layout()
plt.show()

In [ ]:
# --- 10. Correlation matrix ---
feat_cols = [c for c in ['irradiance', 'power', 'ambient_temperature', 'panel_temperature', 'wind_speed'] if c in df.columns]
corr = df[feat_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# --- 11. Observations ---
print("""
Key observations (fill in after running the above cells):

1. Irradiance–power relationship: ...
2. Temperature effects: ...
3. Nighttime behaviour: ...
4. Missing data: ...
5. Seasonal variation: ...
""")